### Rag Pipelines - Data Ingestion Pipeline to Vector DB pipeline

In [2]:
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\chira\AppData\Local\Temp\ipykernel_20796\329546744.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
c:\Users\chira\OneDrive\Desktop\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
### Read all the PDF files in the "pdfs" directory

def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #find all the PDF files recursively in the directory
    pdf_files = list(pdf_dir.rglob("*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in '{pdf_directory}'.")

    for pdf_file in pdf_files:
        print(f"Processing '{pdf_file}'...")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            #add the source metadata to each document
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"Successfully processed '{pdf_file} \n{len(documents)} pages'.")

        except Exception as e:
            print(f"Error processing '{pdf_file}': {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data/pdf_files")

Found 2 PDF files in '../data/pdf_files'.
Processing '..\data\pdf_files\chemistry.pdf'...
Successfully processed '..\data\pdf_files\chemistry.pdf 
25 pages'.
Processing '..\data\pdf_files\science.pdf'...
Successfully processed '..\data\pdf_files\science.pdf 
22 pages'.

Total documents loaded: 47


In [4]:
all_pdf_documents


[Document(metadata={'producer': 'Adobe PDF Library 9.0', 'creator': 'Adobe InDesign CS4 (6.0)', 'creationdate': '2011-01-19T13:38:57-09:00', 'source': '..\\data\\pdf_files\\chemistry.pdf', 'file_path': '..\\data\\pdf_files\\chemistry.pdf', 'total_pages': 25, 'format': 'PDF 1.6', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2011-03-17T11:14:03-08:00', 'trapped': '', 'encryption': 'Standard V2 R3 128-bit RC4', 'modDate': "D:20110317111403-08'00'", 'creationDate': "D:20110119133857-09'00'", 'page': 0, 'source_file': 'chemistry.pdf', 'file_type': 'pdf'}, page_content='What Is In This Chapter?\n1. Chemical characteristics of water \n2. Elements\n3. Compounds\n4. Constituents in water\n5. Biological characteristics of water\n6. Disease and disease transmission\nChapter 2\nBasic Science Concepts\nKey Words\n•\t Aerobic\n•\t Alkalinity\n•\t Anaerobic\n•\t Anion\n•\t Aquatic\n•\t Bacteria\n•\t Cation\n•\t Chemistry\n•\t Colloidal\n•\t Colloidal Solids\n•\t Covalent Bond

### CHUNKING

In [5]:
### Text splitting into chunks

def split_documents(documents, chunk_size=1000,chunk_overlap=200):
    '''Split documents into smaller chunks for better RAG performance'''
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    # Show a sample of the split documents
    if split_docs:
        print("\nSample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs



In [6]:
chunks = split_documents(all_pdf_documents)
chunks

Split 47 documents into 126 chunks.

Sample chunk:
Content: What Is In This Chapter?
1. Chemical characteristics of water 
2. Elements
3. Compounds
4. Constituents in water
5. Biological characteristics of water
6. Disease and disease transmission
Chapter 2
Ba...
Metadata: {'producer': 'Adobe PDF Library 9.0', 'creator': 'Adobe InDesign CS4 (6.0)', 'creationdate': '2011-01-19T13:38:57-09:00', 'source': '..\\data\\pdf_files\\chemistry.pdf', 'file_path': '..\\data\\pdf_files\\chemistry.pdf', 'total_pages': 25, 'format': 'PDF 1.6', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2011-03-17T11:14:03-08:00', 'trapped': '', 'encryption': 'Standard V2 R3 128-bit RC4', 'modDate': "D:20110317111403-08'00'", 'creationDate': "D:20110119133857-09'00'", 'page': 0, 'source_file': 'chemistry.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Adobe PDF Library 9.0', 'creator': 'Adobe InDesign CS4 (6.0)', 'creationdate': '2011-01-19T13:38:57-09:00', 'source': '..\\data\\pdf_files\\chemistry.pdf', 'file_path': '..\\data\\pdf_files\\chemistry.pdf', 'total_pages': 25, 'format': 'PDF 1.6', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2011-03-17T11:14:03-08:00', 'trapped': '', 'encryption': 'Standard V2 R3 128-bit RC4', 'modDate': "D:20110317111403-08'00'", 'creationDate': "D:20110119133857-09'00'", 'page': 0, 'source_file': 'chemistry.pdf', 'file_type': 'pdf'}, page_content='What Is In This Chapter?\n1. Chemical characteristics of water \n2. Elements\n3. Compounds\n4. Constituents in water\n5. Biological characteristics of water\n6. Disease and disease transmission\nChapter 2\nBasic Science Concepts\nKey Words\n•\t Aerobic\n•\t Alkalinity\n•\t Anaerobic\n•\t Anion\n•\t Aquatic\n•\t Bacteria\n•\t Cation\n•\t Chemistry\n•\t Colloidal\n•\t Colloidal Solids\n•\t Covalent Bond

### Embedding & VectorStoreDB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid # for generating unique IDs for each document chunk
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
# Responsible for embeddings
class EmbeddingManager:
    '''Handles document embedding using Sentence Transformer'''

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"): # this model is specifically for changing text into vectors
        """
        Initialize the embedding manager

        Args:
            model_name : HuggingFace model name for sentence embedding
        """
        self.model_name = model_name
        self.model = None # will add the model later
        self._load_model() # _ is for protected function


    def _load_model(self):
        '''Load the sentence transformer model'''
        try:
            print(f"Loading embedding model {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print("Model loaded successfully. Embedding dimension:", self.model.get_sentence_embedding_dimension())
        except Exception as e:
            print(f"Error loading model '{self.model_name}': {e}")
            raise e
        
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        '''Generate embeddings for a list of texts
        Args:
            texts : List of strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        '''

        if not self.model:
            raise ValueError("Model not loaded. Call _load_model() first.")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True) # encode is the fumction used on the text we gave 
        print(f"Embeddings generated successfully with shape: {embeddings.shape}")
        return np.array(embeddings)
    
## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1828.19it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\chira\AppData\Local\Temp\ipykernel_20796\1787645056.py:22: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Model loaded successfully. Embedding dimension:", self.model.get_sentence_embedding_dimension())


### Vector Store

In [9]:
class VectorStore:
    '''Manage Document embedding in a ChromaDB vector store'''
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        '''
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        '''

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        '''Initialize ChromaDB client and collection'''
        try:
            # create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embedding for RAG"}
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        '''
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        '''
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # document content
            documents_text.append(doc.page_content)

            # embedding
            embeddings_list.append(embedding.tolist())

        # add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [15]:
chunks

[Document(metadata={'producer': 'Adobe PDF Library 9.0', 'creator': 'Adobe InDesign CS4 (6.0)', 'creationdate': '2011-01-19T13:38:57-09:00', 'source': '..\\data\\pdf_files\\chemistry.pdf', 'file_path': '..\\data\\pdf_files\\chemistry.pdf', 'total_pages': 25, 'format': 'PDF 1.6', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2011-03-17T11:14:03-08:00', 'trapped': '', 'encryption': 'Standard V2 R3 128-bit RC4', 'modDate': "D:20110317111403-08'00'", 'creationDate': "D:20110119133857-09'00'", 'page': 0, 'source_file': 'chemistry.pdf', 'file_type': 'pdf'}, page_content='What Is In This Chapter?\n1. Chemical characteristics of water \n2. Elements\n3. Compounds\n4. Constituents in water\n5. Biological characteristics of water\n6. Disease and disease transmission\nChapter 2\nBasic Science Concepts\nKey Words\n•\t Aerobic\n•\t Alkalinity\n•\t Anaerobic\n•\t Anion\n•\t Aquatic\n•\t Bacteria\n•\t Cation\n•\t Chemistry\n•\t Colloidal\n•\t Colloidal Solids\n•\t Covalent Bond

In [12]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

### generate the embeddings

embeddings = embedding_manager.generate_embeddings(texts)

### store in the vector database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 126 texts...


Batches: 100%|██████████| 4/4 [00:12<00:00,  3.19s/it]


Embeddings generated successfully with shape: (126, 384)
Adding 126 documents to vector store...
Successfully added 126 documents to vector store
Total documents in collection: 126


### Retriever Pipeline From Vector Store

In [20]:
class RAGRetriever:
    '''Handles query-based retrieval from the vector store'''

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        '''
        Initialize the retriever

        Args:
        vector_store: Vector store containing document embeddings
        embedding_manager: Manager for generating query embeddings
        '''

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        '''
        Retrieve relevant documents for a query

        Args:
            query: the search query
            top_k: Number of top result to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        '''

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")

        # generate query embeddings
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
rag_retriever = RAGRetriever(vectorstore, embedding_manager)
rag_retriever

In [21]:
rag_retriever.retrieve("What are the physical state of matters")

Retrieving documents for query: 'What are the physical state of matters'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 51.37it/s]

Embeddings generated successfully with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_3fa8977b_4',
  'content': 'three large groups that are called the physical states of matter:\nChemical Characteristics of Water\n1 Chemistry – The study of substances \nand the changes they undergo.\n2 Matter – Anything that has weight \nand occupies space.\nIntroduction',
  'metadata': {'subject': '',
   'format': 'PDF 1.6',
   'encryption': 'Standard V2 R3 128-bit RC4',
   'content_length': 238,
   'source': '..\\data\\pdf_files\\chemistry.pdf',
   'moddate': '2011-03-17T11:14:03-08:00',
   'creationDate': "D:20110119133857-09'00'",
   'modDate': "D:20110317111403-08'00'",
   'creationdate': '2011-01-19T13:38:57-09:00',
   'author': '',
   'page': 1,
   'title': '',
   'total_pages': 25,
   'file_type': 'pdf',
   'keywords': '',
   'source_file': 'chemistry.pdf',
   'doc_index': 4,
   'trapped': '',
   'file_path': '..\\data\\pdf_files\\chemistry.pdf',
   'producer': 'Adobe PDF Library 9.0',
   'creator': 'Adobe InDesign CS4 (6.0)'},
  'similarity_score': 0.2629182934761

In [22]:
rag_retriever.retrieve("is a sparrow's song genetically-encoded or learned")

Retrieving documents for query: 'is a sparrow's song genetically-encoded or learned'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 30.77it/s]

Embeddings generated successfully with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_d4735c2f_87',
  'content': 'set of expectations. If the sparrow’s song \nwere indeed genetically encoded, we would \nexpect that a sparrow raised in the nest of \na different species would grow up to sing a \nsparrow song like any other member of its own species. But if, instead, the sparrow’s \nsong were learned as a chick, raising a sparrow in the nest of another species should \nproduce a sparrow that sings a non-sparrow song. Because they generate different \nexpected observations, these ideas are testable. A scientific idea may require a lot of \nreasoning to work out an appropriate test, may be difficult to test, may require the \ndevelopment of new technological tools to test, or may require one to make indepen-\ndently testable assumptions to test—but to be scientific, an idea must be testable, \nsomehow, someway.\nIf an explanation is equally compatible with all possible observations, then it is not \ntestable and hence, not within the reach of science. This is fr

### Integration of VectorDB Context pipeline with LLM output

In [27]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initializing the groq LLM
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key=groq_api_key, model_name="llama-3.1-8b-instant", temperature=0.1, max_tokens=1024)

# Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    ## retrieve context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."

    ## generate the answer using Groq LLM
    prompt = f"""Use the following context to answer the question concisely. You are a teacher and you have to explain the student with clear language and explanation.
Context: {context}
Question: {query}

ANSWER:"""

    response = llm.invoke(prompt)
    return response.content

In [28]:
answer = rag_simple("how is the strength of a solution determined", rag_retriever,llm)
print(answer)


Retrieving documents for query: 'how is the strength of a solution determined'
Top K: 3, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 39.93it/s]

Embeddings generated successfully with shape: (1, 384)
Retrieved 2 documents (after filtering)


The strength of a solution is determined by the amount of solute dissolved in a certain amount of water.


### Enhanced RAG Pipeline Features

In [29]:
# Enhanced RAG Pipeline Features

def rag_advanced(query, reriever, llm, top_k=5, min_score = 0.2, return_context = False):
    '''
    RAG pipeline with some extra feature i wanted to implement:
    - Return answer, sources, confidence score, and optionally full context
    '''

    results = reriever.retrieve(query, top_k=top_k, score_threshold = min_score)
    if not results:
        return {'answer': 'No relevant context found', 'sources': [], 'confidence':0.0, 'context': ''}
    
    # Prepare context and source
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'

    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f'''Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer: '''
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer' : response.content,
        'sources': sources,
        'confidence': confidence

    }

    if return_context:
        output['context'] = context
    return output


#Example Usage
result = rag_advanced("how do we get ions?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print('Answer: ', result['answer'])
print('Sources: ', result['sources'])
print('Confidence: ', result['confidence'])
print('Context Preview: ', result['context'][:300])



Retrieving documents for query: 'how do we get ions?'
Top K: 3, Score Threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.43it/s]

Embeddings generated successfully with shape: (1, 384)
Retrieved 2 documents (after filtering)


Answer:  When molecules dissolve in water, the atoms making up the molecules come apart in the water, a process called ionization. This results in the formation of charged atoms called ions, which are positively charged (cations) and negatively charged (anions).
Sources:  [{'source': 'chemistry.pdf', 'page': 8, 'score': 0.25627344846725464, 'preview': 'and can dissolve a number of substances \n(solids, liquids, and gases), many of which \nmust be removed during treatment to make \nwater safe to drink. \nSolutions and Suspensions\nA solution may be colored, but it will be transparent, not cloudy. The solute will \nremain uniformly distributed throughout ...'}, {'source': 'chemistry.pdf', 'page': 8, 'score': 0.12128305435180664, 'preview': 'are always positively charged ions and negatively charged ions when ionization oc\xad\ncurs. The positively charge ions are called cations13, and the negatively charged ions \nare called anions14.\nExamples of Ionization\nA good example is the ionizat